# Mushroom Confusion Diagnosis — Run 5, Grad-CAM, Run 6 (resolution)


## Objective

Run 3/4 showed mushroom stuck at ~78-85% val_accuracy with a large train/val gap (~99.9% vs
~84%) — textbook overfitting signature. Run 5 tested that hypothesis directly: add regularization
(stronger augmentation, weight decay, label smoothing) and see if the gap closes. It didn't. This
notebook documents Run 5, the corrected comparison against Run 4, and a Grad-CAM investigation
into *why* — which points at a different diagnosis than overfitting.

Kept separate from `07_training_run_log.ipynb` since this is a focused investigation with
generated diagnostic images, not a routine run log entry.


## What changed for Run 5

Three cheap, standard regularizers, all newly config-driven (both `build_loss` and
`build_optimizer` already accepted `**kwargs`, so no new plumbing needed beyond reading them
from `training_config` in `scripts/train_baseline.py`):

- `augmentation_preset: light → medium` in `configs/mushroom.yaml` (rotation + stronger color
  jitter, on top of the existing flip)
- `weight_decay: 0.05` (up from AdamW's implicit 0.01 default)
- `label_smoothing: 0.1` (softens target confidence in the cross-entropy loss)

Also fixed an unrelated infrastructure problem discovered during Run 4: `scripts/train_truba_cpu.sbatch`
now requests `--exclusive` — Run 4 shared its node with another job (`CPUAlloc=61` when we only
asked for 56) and epoch times swung between 2900-4760s as a result. Run 5 confirms the fix: epoch
times are flat at ~950s throughout (see the chart below).


![Mushroom Run 5 metrics](assets/mushroom_run5_metrics.png)


## Run 4 vs Run 5 — corrected comparison

The quick read from the W&B charts alone made Run 5 look roughly equivalent to Run 4. Pulling the
exact numbers from both runs' best epoch (by `val_macro_f1`) tells a clearer story:

| | Run 4 (light aug, no weight decay) | Run 5 (medium aug + weight_decay + label_smoothing) |
|---|---|---|
| Best epoch | 18 | 24 |
| val_accuracy | **84.80%** | 83.27% |
| val_macro_f1 | **0.8126** | 0.7911 |
| train_accuracy (same epoch) | 99.93% | 99.95% |
| train/val accuracy gap | ~15.1 pts | ~16.7 pts |

**Regularization made things slightly worse, not better, on every metric that matters, and the
train/val gap didn't shrink either** — train accuracy is still ~99.9%+ regardless of augmentation
strength or weight decay. `val_loss`'s absolute value did go up in Run 5 (1.5 vs 0.88 in Run 4),
but that's an artifact of label smoothing changing the loss function's floor (it never lets loss
approach zero, even for perfect predictions) — not a real regression, and not comparable across
the two runs directly. Accuracy and macro-F1 are what's comparable, and both point the same way.

**This is the actual finding that matters**: if regularization aimed at "the model is
memorizing the training set" doesn't move the needle at all, the working hypothesis
("overfitting due to insufficient regularization") is probably wrong, or at least incomplete.


## The confused-pairs pattern

Run 5's top-10 confused pairs (from `outputs/reports/mushroom_resnet50_eval_report.json`):

| true → predicted | count |
|---|---|
| Fomitopsis pinicola → Fomitopsis mounceae | 64 |
| Fomes fomentarius → Fomitopsis betulina | 31 |
| Amanita muscaria → Amanita persicina | 30 |
| Evernia prunastri → Evernia mesomorpha | 25 |
| Fomitopsis pinicola → Fomes fomentarius | 24 |
| Fomes fomentarius → Ganoderma applanatum | 21 |
| Parmelia sulcata → Hypogymnia physodes | 21 |
| Pleurotus pulmonarius → Pleurotus ostreatus | 21 |
| Xanthoria parietina → Vulpicida pinastri | 21 |
| Leccinum scabrum → Leccinum aurantiacum | 18 |

**Every single one of these pairs is a same-genus confusion**: *Fomitopsis* with *Fomitopsis*,
*Amanita* with *Amanita*, *Evernia* with *Evernia*, *Pleurotus* with *Pleurotus*, *Leccinum* with
*Leccinum* — the two *Fomitopsis pinicola* rows and the *Fomes fomentarius* → *Ganoderma
applanatum* row are all bracket/polypore fungi that look alike even to the model apparently
across genus lines too. This is not noise — it's a consistent, taxonomically coherent error
pattern, which is a very different signal than "the model overfit the training set" would
produce (that would look like more random, spread-out confusion, not concentrated on
closely-related species pairs).

This reframes the question from *"how do we stop the model from memorizing?"* to *"can a
224×224 ResNet-50 tell these specific look-alike species apart at all, and if not, why?"*
— which is exactly what Grad-CAM can help answer: is the model looking at the right part of the
image and still failing (a genuine fine-grained discrimination limit), or is it looking
somewhere irrelevant (a fixable data/pipeline issue)?


## Grad-CAM: where is the model looking when it gets these wrong?

Implemented in `src/explainability/gradcam.py` — a standard Grad-CAM (Selvaraju et al., 2017):
hook the last conv block (`model.backbone.layer4`), backprop the predicted class's score, weight
the activation channels by their gradients, and overlay the resulting heatmap on the input image.

For each of the top 5 confused pairs, loaded the Run 5 best checkpoint, ran inference over the
full validation set, found actual misclassified examples (`true_label == A, predicted_label ==
B`), and ran Grad-CAM on 3 random examples per pair. Red/yellow = where the model's prediction is
most sensitive to; blue = ignored.


![Grad-CAM: Fomitopsis pinicola misclassified as Fomitopsis mounceae](assets/gradcam_Fomitopsis_pinicola_Fomitopsis_mounceae.png)


![Grad-CAM: Fomes fomentarius misclassified as Fomitopsis betulina](assets/gradcam_Fomes_fomentarius_Fomitopsis_betulina.png)


![Grad-CAM: Amanita muscaria misclassified as Amanita persicina](assets/gradcam_Amanita_muscaria_Amanita_persicina.png)


![Grad-CAM: Evernia prunastri misclassified as Evernia mesomorpha](assets/gradcam_Evernia_prunastri_Evernia_mesomorpha.png)


![Grad-CAM: Fomitopsis pinicola misclassified as Fomes fomentarius](assets/gradcam_Fomitopsis_pinicola_Fomes_fomentarius.png)


## Reading the Grad-CAM results

**In all 15 examples across all 5 pairs, the model's attention sits squarely on the fungus or
lichen itself** — never on the bark, snow, moss, hand, or background. There is no example of the
model "cheating" by keying off an irrelevant cue. That rules out the most easily-fixable
explanation (a background/shortcut-learning artifact) and supports the taxonomic-confusion
reading: **the model is looking at exactly the right structures and still can't separate these
species.**

A secondary pattern in the confidence scores is worth noting: the *Fomitopsis pinicola ↔
mounceae* and *Evernia prunastri ↔ mesomorpha* pairs get consistently high-confidence wrong
predictions (0.76-0.96) with attention covering the whole fruiting body/thallus — the model isn't
uncertain, it's confidently wrong, which is consistent with these species being genuinely
close to indistinguishable at this resolution/scale. The *Fomes fomentarius ↔ Fomitopsis
betulina* pair instead shows lower, more varied confidence (0.41-0.96) and attention sometimes
concentrated on a smaller sub-region rather than the whole structure — a messier, more
borderline case, possibly a harder pair even for the correct-answer cases, or images where the
diagnostic surface (pore structure, which isn't very visible in bracket-fungus photos taken from
this angle) simply isn't captured by the photo.

**What this does and doesn't tell us:**
- Does *not* support: more/heavier regularization (Run 5 already tested this, no effect), or
  suspecting a background/data-leakage artifact (Grad-CAM shows none).
- Does support treating this as a genuine fine-grained visual discrimination problem, where the
  next things worth testing (in order) are things that affect how much visual detail the model
  can actually see and use, not things that fight overfitting:
  1. **Higher input resolution** (384×384, already benchmarked as an option in Sprint 3) — 224px
     may be discarding the fine surface/texture detail that separates e.g. *Fomitopsis pinicola*
     from *mounceae*.
  2. **A different architecture** (EfficientNet-B3, per the Sprint 4 research note) — compare
     whether it makes the *same* genus-level mistakes. If yes, this is a dataset/resolution
     ceiling, not a ResNet-50-specific weakness. If no, architecture matters more than expected
     here.
  3. Only after 1-2: consider whether these specific confused species need targeted extra data,
     rather than assuming more data across the board would help evenly.


## Run 6 — resolution experiment (224px → 384px)

Testing priority 1 from the conclusion above: does more input resolution recover the fine
surface detail these look-alike species need? Isolated as the *only* change from Run 4
(`configs/mushroom.yaml`: `image_size: 224 → 384`, augmentation/weight_decay/label_smoothing
reverted to Run 4's settings — light augmentation, no extra weight decay, no label smoothing).
Same command: `sbatch scripts/train_truba_cpu.sbatch configs/mushroom.yaml 30 "" 300`.

- Duration: 89055.0s (1 day 44 min) — epoch time roughly tripled vs 224px (~2870-3330s vs
  ~950s), as expected from ~2.94x more pixels per image
- **Best epoch: 24** (val_macro_f1=0.8511, val_accuracy=88.10%)
- train_accuracy 99.99% at the best epoch — still fully memorizing the training set;
  384px didn't reduce overfitting, it just raised the val ceiling alongside it


![Mushroom Run 6 (384px) metrics](assets/mushroom_run6_384px_metrics.png)


### 224px vs 384px — real improvement, but partial

| | Run 5 (224px, regularized) | Run 6 (384px, Run-4 settings) |
|---|---|---|
| val_accuracy | 83.27% | **88.10%** |
| val_macro_f1 | 0.7911 | **0.8511** |
| train_accuracy (best epoch) | 99.95% | 99.99% |
| epoch duration | ~950s | ~2900-3300s (~3x) |

Clear, genuine gain: **+4.8 points accuracy, +0.06 macro-F1**. Resolution was worth trying.

### But the same confused pairs — just less often, not gone

| pair | Run 5 (224px) count | Run 6 (384px) count | change |
|---|---|---|---|
| Fomitopsis pinicola → Fomitopsis mounceae | 64 | 40 | -37.5% |
| Amanita muscaria → Amanita persicina | 30 | 14 | -53.3% |
| Fomes fomentarius → Fomitopsis betulina | 31 | 16 | -48.4% |
| Evernia prunastri → Evernia mesomorpha | 25 | 17 | -32.0% |
| Fomitopsis pinicola → Fomes fomentarius | 24 | 17 | -29.2% |
| Xanthoria parietina → Vulpicida pinastri | 21 | 18 | -14.3% |

Every one of Run 5's top pairs shows real improvement at 384px — resolution genuinely helps
separate these species, supporting the "fine detail lost at 224px" part of the diagnosis.

**But the pattern didn't disappear, it reshuffled.** Run 6's own top-10 confused pairs are
*still* exactly the same genus clusters (Fomitopsis, Evernia, Amanita, Xanthoria/Vulpicida),
plus a **new bidirectional pair** that wasn't prominent before: `Leccinum versipelle ↔
Leccinum aurantiacum` (17 and 16 misclassifications *in both directions*), and a third
Fomitopsis pair (`pinicola → betulina`, 15). `Fomitopsis mounceae` is still the single worst
class by F1 (0.504) even at 384px. Higher resolution didn't eliminate the genus-level ceiling,
it just moved where on that ceiling the remaining errors concentrate.

**Reading this**: resolution is a real, worthwhile lever, but not a full fix on its own — and
its cost is steep. At 300/class, full-dataset extrapolation (689,520 images, ~13.6x more) would
put a 384px run at roughly 2-3 *weeks* of wall-clock time on this CPU cluster, which isn't
practical, especially without mixed precision (still a no-op on CPU, see
`scripts/train_baseline.py` — this only matters on CUDA, not available here).


### Next: EfficientNet-B3 comparison, at 224px

Priority 3 from the original list, promoted up given 384px's cost/benefit: train
EfficientNet-B3 at 224px (not 384, to keep this experiment's cost controlled and isolate
architecture as the one variable) and check whether it makes the *same* genus-level mistakes.

- **If yes** (same Fomitopsis/Evernia/Amanita/Xanthoria confusions dominate): this is a
  dataset/subset-scale ceiling, not a ResNet-50-specific weakness — resolution and architecture
  both help partially but neither alone resolves it, and the real fix is more data specifically
  for these species, not more model changes.
- **If no** (EfficientNet separates these pairs meaningfully better): architecture matters more
  here than expected, worth pursuing further (e.g. EfficientNet at 384px too, if the accuracy
  gain justifies the cost).


## Full confusion matrix — and a correction to the genus-confusion narrative

Run 6's 384px checkpoint, full validation set (15,616 images), all 169 species — not just the
top-10 confused pairs this time. Computed with `scratch_confusion_matrix.py`, not saved to the
repo (one-off analysis, not a reusable script).

- **Species accuracy: 88.10%** — matches Run 6's reported number
- **Genus accuracy: 91.02%** (derived by mapping both true and predicted species to their genus)
- Of the **1,859 species-level errors**, only **456 (24.5%) still got the genus right**

**This is a real correction to the earlier framing.** The top-10 confused-pairs list (used in
every run's analysis so far) is dominated by same-genus pairs because those individual pairs
have the highest counts — but that list is not representative of errors *in aggregate*. Looking
at all 1,859 errors, **75.5% are actually cross-genus mistakes**, not within-genus ones. The
genus-clustering story was real but overstated: same-genus confusion is the single most visible
pattern, not the dominant one.

This matters directly for the hierarchical-loss hypothesis below: if only ~24.5% of species
errors are within-genus, then even a hypothetically *perfect* genus classifier could only ever
fix, at best, a quarter of the current errors — the other three-quarters are mistakes between
genera that genus-awareness doesn't help with by construction.


![Mushroom full species-level confusion matrix, sorted by genus](assets/mushroom_confusion_matrix_species_full.png)


![Mushroom genus-level confusion matrix](assets/mushroom_confusion_matrix_genus.png)


## EfficientNet-B3 comparison — same genus-level mistakes, slightly worse overall

Trained EfficientNet-B3 at 224px (not 384, to isolate architecture and control cost) on both
datasets, identical settings to Run 4 (mushroom) and the standing flower config otherwise:

| | ResNet-50 | EfficientNet-B3 |
|---|---|---|
| Mushroom val_accuracy | **84.80%** (Run 4) | 83.68% |
| Mushroom val_macro_f1 | **0.8126** | 0.7983 |
| Flower val_accuracy | **98.80%** (Run 3) | 98.78% |
| Flower val_macro_f1 | **0.9882** | 0.9867 |

ResNet-50 comes out slightly ahead on both datasets — not the "newer architecture wins" result
one might expect by default. More importantly, EfficientNet-B3's top confused pairs on mushroom
are the **same genus clusters**: `Fomitopsis pinicola → Fomitopsis mounceae` (37),
`Xanthoria parietina → Vulpicida pinastri` (37), `Amanita muscaria → Amanita persicina` (32),
`Fomes fomentarius → Ganoderma applanatum` (30), `Evernia prunastri → Evernia mesomorpha` (21).

**This directly answers the question the comparison was designed to ask**: since a different
architecture produces the same confusion pattern, it isn't a ResNet-50-specific weakness — it's
a property of the dataset/subset at this scale. Combined with the confusion-matrix finding above,
switching architectures was not worth pursuing further; effort went into the hierarchical model
instead (below), built on ResNet-50 since it's the stronger baseline.


## Hierarchical (genus+species) multi-task model

Approach: one shared ResNet-50 backbone, two linear heads (`genus_head`, `species_head`),
`total_loss = genus_loss + species_loss` (`src/models/hierarchical_resnet.py`,
`scripts/train_hierarchical.py` — a separate training script since the two-logit-tensor output
doesn't fit the shared single-task `src/training/engine.py` path used everywhere else). Genus
labels are derived from species names (first word) via a lookup tensor built once at startup —
no dataset changes needed. Kept at Run 4's settings otherwise (224px, light augmentation, same
`ReduceLROnPlateau`/early-stopping patience) specifically to isolate the hierarchical loss as the
one changed variable, per the plan: confusion matrix first (above), then hierarchical loss with
the LR schedule untouched, and only tune the schedule further if this step showed a clear win.


![Hierarchical genus+species training metrics](assets/mushroom_hierarchical_metrics.png)


### Results — a small species-accuracy gain, but the confused pairs didn't improve

| | Run 4 (single-head, 224px) | Hierarchical (genus+species, 224px) |
|---|---|---|
| val_species_accuracy | 84.80% | 85.3% (best epoch 29) |
| val_species_macro_f1 | 0.8126 | 0.8156 |
| val_genus_accuracy | — (not modeled) | 88.7% (best epoch) |
| Fomitopsis pinicola → mounceae | 64 | **73 (worse)** |
| Xanthoria parietina → Vulpicida pinastri | 21 | **31 (worse)** |
| Amanita muscaria → persicina | — | 32 |

Species accuracy and macro-F1 both improved, but only marginally (+0.5pt accuracy, +0.003
macro-F1) — and **the flagship same-genus confused pairs got worse in raw count**, not better.
The hypothesis going in was "teaching the model genus explicitly will push its features to
separate genus-mates better" — that didn't clearly happen here.

**Why, in light of the confusion-matrix finding above**: genus accuracy plateaus at 88.7%
here too (`train_genus_loss` collapses to ~0.001 by epoch 15 — the auxiliary task overfits the
300/class subset just as hard as species did). An auxiliary task that itself overfits doesn't
give the shared backbone a cleaner signal to learn from. And since only ~24.5% of species errors
are within-genus in the first place, even a *working* hierarchical signal has a low ceiling on
how much it could help — most of the error is genus confusion or unrelated mistakes that
genus-awareness doesn't target by construction.

**Worst-10 species (hierarchical model)** — same names as every prior run (Fomitopsis mounceae,
Boletus reticulatus, Leccinum aurantiacum, Amanita persicina, …), consistent with a stable,
subset-scale ceiling rather than noise from any one run.

**Conclusion**: the hierarchical loss is not the fix. Combined with Run 5 (regularization: no
help), Run 6 (resolution: real but partial help), and EfficientNet-B3 (architecture: no help),
four different interventions have now been tried against this ceiling. The one lever not yet
tested at scale is the one the confusion-matrix analysis points at most directly — more real
data (the full 4,080/class dataset, not this 300/class subset) — since every architecture/loss/
regularization change so far has run into the same wall while data quantity was held fixed.


## Veri Kalitesi Bulguları — "balanced 4080/class" dataset gerçekte dengeli değil

Bu bölüm dört ayrı ama birbirine bağlı bulguyu birleştiriyor. Sıra önemli: önce leakage
kontrolü (aşağıdaki bulguların güvenilir olduğunu doğrulamak için), sonra üç liste.

### 0) Leakage kontrolü — TEMİZ

`train.csv`/`val.csv`/`test.csv`'deki tüm 104,088 benzersiz dosya yolu için MD5 içerik hash'i
hesaplandı (`scratch_leakage_check.py`, silindi — tek seferlik analiz):

- **0 byte-özdeş kopya** herhangi bir dosya çiftinde
- **0 split-arası (train↔val, train↔test, val↔test) sızıntı**
- **0 farklı-etiket-altında-aynı-görsel** (etiketleme hatası) örneği

Yani aşağıdaki bulgular sızıntıdan kaynaklanmıyor — gerçek bir veri kalitesi meselesi. (Not: bu
kontrol sadece *byte-birebir* kopyaları yakalar; Sprint 1'in orijinal "near-duplicate" endişesi
— aynı çekimin hafif kırpılmış/işlenmiş halleri — perceptual hashing gerektirir, bu daha zayıf
bir sızıntı riski olarak hâlâ açık kalıyor.)


### a) Sahte-dengeli / düşük-benzersiz sınıflar

`train.csv`'de her tür tam olarak **4080 satır** — ama bu satırlar aynı görselin defalarca
tekrarıdır. Gerçek benzersiz görsel sayısı türden türe **147 ile 4080 arasında**, 27.7 kat fark:

| Tür | Benzersiz görsel | Tekrar oranı |
|---|---|---|
| Psilocybe caerulescens | 147 | 27.8 |
| Panaeolus papilionaceus | 154 | 26.5 |
| Volvopluteus gloiocephalus | 161 | 25.3 |
| Cryptoporus volvatus | 162 | 25.2 |
| Galerina marginata | 164 | 24.9 |
| Chlorophyllum molybdites | 165 | 24.7 |
| Laccaria ochropurpurea | 165 | 24.7 |
| Coprinopsis lagopus | 169 | 24.1 |
| Psathyrella candolleana | 170 | 24.0 |
| Gymnopilus luteofolius | 171 | 23.9 |
| Ganoderma oregonense | 171 | 23.9 |
| Phaeophyscia orbicularis | 171 | 23.9 |
| Suillus grevillei | 171 | 23.9 |
| Trametes gibbosa | 173 | 23.6 |
| Coltricia perennis | 174 | 23.4 |

- **169 türün 110'i (65%) 300'den az** benzersiz görsele sahip
- **169 türün 49'i (28%) 200'den az** benzersiz görsele sahip
- Ortanca (median) benzersiz görsel sayısı: **227** — "4080/sınıf dengeli" iddiasının aksine

Bu, `mushroom.yaml`'ın en baştaki yorumundaki "balanced (std=0 in Sprint 1)" ifadesinin **yanlış
olmadığını ama eksik** olduğunu gösteriyor — Sprint 1'in ölçtüğü satır sayısı gerçekten dengeliydi,
ama satır sayısı ≠ gerçek veri çeşitliliği. Sprint 1'in kendi flaglediği "near-duplicate question"
tam olarak buymuş, şimdi kesin olarak doğrulandı.

**Bunun `--subset-per-class 300` deneylerimiz için sonucu**: Trametes gibbosa gibi 173 benzersiz
görselli bir tür için "300/sınıf" subset'i zaten mevcut tüm çeşitliliği kapsıyordu (üstüne az
miktarda tekrar ekliyordu). Bu yüzden **"tam veri setine geçmek" bu türler için hiçbir ek bilgi
getirmeyecek** — sadece zaten-zengin türler (Fomitopsis pinicola, Xanthoria, Fomes fomentarius,
Amanita muscaria gibi) tam veriden gerçekten faydalanabilir.


### b) Veri kıtlığından kötü performans gösteren sınıflar

Eşik: <300 benzersiz görsel VE F1<0.80 (Run 6, 384px checkpoint'i, tam validation seti, tüm 169
tür için hesaplanan gerçek precision/recall/F1 — sadece top-10 değil). **26 tür** bu
eşiği geçiyor:

| Tür | Benzersiz görsel | Val support | F1 |
|---|---|---|---|
| Fomitopsis mounceae | 187 | 40 | 0.5039 |
| Boletus reticulatus | 176 | 38 | 0.5517 |
| Trametes ochracea | 186 | 40 | 0.6279 |
| Trametes gibbosa | 173 | 37 | 0.6486 |
| Amanita persicina | 184 | 40 | 0.6512 |
| Phlebia tremellosa | 200 | 43 | 0.6575 |
| Lycoperdon pyriforme | 210 | 45 | 0.6588 |
| Volvopluteus gloiocephalus | 161 | 35 | 0.7059 |
| Daedaleopsis tricolor | 194 | 41 | 0.7179 |
| Phellinus igniarius | 235 | 50 | 0.7304 |
| Armillaria borealis | 189 | 41 | 0.7312 |
| Psilocybe ovoideocystidiata | 175 | 38 | 0.7342 |
| Trametes betulina | 203 | 44 | 0.7442 |
| Imleria badia | 206 | 44 | 0.7579 |
| Amanita amerirubescens | 235 | 50 | 0.7579 |
| Agaricus augustus | 236 | 50 | 0.7647 |
| Merulius tremellosus | 274 | 59 | 0.7669 |
| Trametes hirsuta | 196 | 42 | 0.7692 |
| Ganoderma curtisii | 181 | 39 | 0.7714 |
| Cryptoporus volvatus | 162 | 35 | 0.7812 |
| Agaricus xanthodermus | 212 | 46 | 0.7826 |
| Armillaria tabescens | 246 | 53 | 0.7835 |
| Leucoagaricus leucothites | 180 | 38 | 0.7838 |
| Armillaria mellea | 227 | 49 | 0.7872 |
| Amanita calyptroderma | 228 | 49 | 0.7901 |
| Psilocybe caerulescens | 147 | 32 | 0.7937 |

Bu liste, Run 3-6'nın worst-classes tablolarında sürekli gördüğümüz isimlerle (Fomitopsis
mounceae, Boletus reticulatus, Trametes türleri, Amanita persicina) neredeyse birebir örtüşüyor
— tesadüf değil, aynı kök nedenin farklı görünümleri.


### c) Veri zengin olduğu halde gerçek görsel benzerlikten kötü performans gösteren çiftler

Önce "veri-zengin (≥1000 benzersiz) ama kendi F1'i düşük tür" aradım — **böyle bir tür yok**.
Veri-zengin türler kendi performanslarında iyiler; sorun onların **kardeş türlerini** karıştırması.
Doğru soru: "her iki tarafı da veri-zengin olan ama yine de birbirine karışan çiftler hangileri?"

Eşik: her iki taraf da ≥1000 benzersiz görsel VE ≥5 kez karışmış (tam confusion matrix'ten,
sadece top-10 değil, 1084 karışan çiftin tamamından filtrelendi):

| Gerçek tür | Tahmin edilen | Kaç kez | Gerçek benzersiz | Tahmin benzersiz |
|---|---|---|---|---|
| Fomitopsis pinicola | Fomes fomentarius | 17 | 3562 | 3673 |
| Fomes fomentarius | Fomitopsis betulina | 16 | 3673 | 1111 |
| Fomitopsis pinicola | Fomitopsis betulina | 15 | 3562 | 1111 |
| Fomes fomentarius | Fomitopsis pinicola | 12 | 3673 | 3562 |
| Hypogymnia physodes | Parmelia sulcata | 12 | 1514 | 1847 |
| Leccinum scabrum | Boletus edulis | 9 | 1185 | 1027 |
| Parmelia sulcata | Hypogymnia physodes | 5 | 1847 | 1514 |

Bu **7 çiftin hepsi** ya *Fomitopsis/Fomes* (braket mantarları) kümesi ya da *Hypogymnia/Parmelia*
(liken) çifti — ikisi de bol veriye rağmen birbirine gerçekten görsel olarak çok benziyor. Bu,
"daha fazla veri" ile çözülmeyecek bir sınır — Grad-CAM analizimiz de (yukarıda) bu çiftlerde
modelin doğru yere baktığını ama yine de ayırt edemediğini zaten göstermişti.


### d) Taksonomik olarak fungi olmayan / liken sınıflar

169 türün **18'i liken** (14 cins: Cetraria, Cladonia, Evernia, Graphis, Hypogymnia, Lobaria,
Parmelia, Peltigera, Phaeophyscia, Physcia, Platismatia, Pseudevernia, Vulpicida, Xanthoria) ve
**6'sı** ne liken ne klasik "şapkalı mantar" — performanslarıyla birlikte:

| Tür | Benzersiz görsel | Val support | F1 |
|---|---|---|---|
| Physcia adscendens | 259 | 55 | 0.748 |
| Pseudevernia furfuracea | 312 | 67 | 0.7727 |
| Phaeophyscia orbicularis | 171 | 36 | 0.7848 |
| Evernia mesomorpha | 497 | 106 | 0.8426 |
| Graphis scripta | 237 | 51 | 0.8475 |
| Evernia prunastri | 1024 | 219 | 0.9087 |
| Vulpicida pinastri | 604 | 129 | 0.9091 |
| Hypogymnia physodes | 1514 | 324 | 0.9214 |
| Sarcosoma globosum | 196 | 42 | 0.9286 |
| Calycina citrina | 418 | 90 | 0.929 |
| Peltigera praetextata | 203 | 44 | 0.9333 |
| Sarcoscypha austriaca | 273 | 59 | 0.9365 |
| Lycogala epidendrum | 175 | 38 | 0.9367 |
| Peltigera aphthosa | 194 | 42 | 0.9425 |
| Parmelia sulcata | 1847 | 396 | 0.944 |
| Platismatia glauca | 281 | 60 | 0.944 |
| Cladonia stellaris | 251 | 54 | 0.9455 |
| Cladonia fimbriata | 218 | 47 | 0.9485 |
| Cetraria islandica | 301 | 65 | 0.9552 |
| Lobaria pulmonaria | 434 | 93 | 0.9574 |
| Cladonia rangiferina | 175 | 38 | 0.9589 |
| Xanthoria parietina | 4080 | 874 | 0.9698 |
| Rhytisma acerinum | 561 | 120 | 0.9712 |
| Chlorociboria aeruginascens | 238 | 51 | 0.9901 |

Not: `Lycogala epidendrum` bir **slime mold** — Fungi krallığından bile değil, Protista. Diğerleri
(Rhytisma, Sarcoscypha, Sarcosoma, Calycina, Chlorociboria) gerçek mantar ama yaprak lekesi/küçük
disk/odun lekesi formunda, "şapkalı mantar" değil.

**Karar önerisi**: Bu sınıfları veri setinden çıkarmak zorunlu değil — istersen kalsınlar, ama
README/rapor'da "bu sınıf taksonomik olarak [gilled] fungi değil, veri kaynağının kapsamından
sızmış, pipeline'ın hata analiz araçları bunu tespit etti" şeklinde belgelemek yeterli. Bu bir
"bug fix" değil, dataset'in kapsamı hakkında şeffaf bir **gözlem**. Performansları da (çoğu
0.84-0.99 F1, oldukça iyi) veri setinden çıkarılmaları için acil bir teknik gerekçe sunmuyor —
liken sınıfının en kötüsü (Physcia adscendens, 0.748) bile mantar sınıflarının çoğundan iyi.


### Sentez

Dört bulgu birbirini tamamlıyor:
1. **Sızıntı yok** — bulgular güvenilir.
2. **169 türün üçte ikisi (110/169) gerçekte 300'den az benzersiz fotoğrafla temsil ediliyor** —
   "4080/sınıf dengeli" satır sayısı bunu gizliyor.
3. **26 tür**, düşük veri + düşük F1 ile açıkça **veri kıtlığından** zarar görüyor — bunlar için
   gerçekten daha fazla *benzersiz* veri toplamak (mevcut 4080 satırı büyütmek değil, yeni
   fotoğraf bulmak) tek gerçek çözüm.
4. **7 çift**, bol veriye rağmen birbirine karışıyor — **gerçek taksonomik/görsel zorluk**, ne
   augmentation ne weight decay ne hiyerarşik loss ne de "daha fazla veri" bunu çözer; Grad-CAM
   zaten bunu göstermişti.
5. **24 tür liken/fungi-olmayan** — teknik bir sorun değil, dataset kapsamının bir gerçeği;
   belgelemek yeterli, performansları zaten iyi.

**Pratik sonuç**: "tam veri setine geçelim" planı artık çok daha hedefli hale geldi — 110 türün
gerçek faydası olmayacak (zaten subset'te tüm verilerini görüyorlardı), ama geri kalan ~59 tür
(özellikle Fomitopsis pinicola, Xanthoria, Fomes fomentarius, Amanita muscaria gibi zaten
veri-zengin olanlar) tam veriden gerçekten faydalanabilir. Yeni bir "daha fazla veri" denemesi
yapılacaksa, bunu **sadece gerçekten veri-zengin sınıflarda** test etmek, kıt sınıflarda değil,
daha anlamlı bir sonuç verecektir.


## What changed for Run 7 — scarcity-aware augmentation

Targeted intervention motivated directly by the Veri Kalitesi Bulguları section above: instead
of blanket regularization (Run 5, which hurt data-rich classes without helping scarce ones),
apply `heavy` augmentation *only* to the 26 species from list (b) (<300 real unique images,
F1<0.80) — everything else keeps Run 6's `light` preset unchanged. Implemented via
`ScarcityAwareTransform` (`src/data/transforms.py`) — `ImageClassificationDataset` now checks
per-sample label and picks the transform accordingly (`src/data/dataset.py`). Isolated against
Run 6: only the augmentation-per-class-group changed, resolution/optimizer/LR schedule/patience
all identical (`configs/mushroom_scarcity_aware.yaml`).


## Run 7 results — worse, not better, and the targeted classes collapsed

| | Run 6 (uniform light, 384px) | Run 7 (scarcity-aware heavy, 384px) |
|---|---|---|
| val_accuracy (best epoch) | **88.10%** | 86.59% |
| val_macro_f1 | **0.8511** | 0.8029 |
| train_accuracy (best epoch) | 99.99% | 99.95% |
| best epoch | 24 | 27 |

Worse on every metric that matters. But the more important signal is *which* classes got worse
— exactly the 26 that received heavy augmentation:

| Class (got `heavy` aug) | Run 7 F1 | Run 7 recall |
|---|---|---|
| Boletus reticulatus | **0.095** | **0.053** |
| Lycoperdon pyriforme | 0.200 | 0.111 |
| Daedaleopsis tricolor | 0.250 | 0.146 |
| Fomitopsis mounceae | 0.308 | 0.200 |
| Phellinus igniarius | 0.317 | 0.200 |
| Amanita persicina | 0.320 | 0.200 |

`Boletus reticulatus`'s recall of 0.053 means the model correctly identifies it in essentially
**2 out of 38** validation images — a near-total collapse. These are the *same* species Run 6
already struggled with (they were list (b) precisely because Run 6's F1 was <0.80 for them), but
heavy augmentation made every one of them dramatically worse, not better.

The new top confused pairs confirm the mechanism — the scarce classes are being absorbed into
their data-rich, visually-similar siblings:

| true → predicted | count | true's group |
|---|---|---|
| Daedaleopsis tricolor → Daedaleopsis confragosa | 31 | scarce → rich |
| Lycoperdon pyriforme → Apioperdon pyriforme | 30 | scarce → rich |
| Boletus reticulatus → Boletus edulis | 23 | scarce → rich |
| Fomitopsis mounceae → Fomitopsis pinicola | 20 | scarce → rich |
| Phellinus igniarius → Phellinus tremulae | 20 | scarce → (unweighted) |

**Why this likely backfired**: `heavy` includes ±25° rotation, strong color jitter, and
`RandomPerspective(distortion_scale=0.2)`. Applied to a class with only ~150-300 real photos
(each already reused ~20-25x per epoch just to reach the nominal 4080 rows), this pushes the
already-thin real signal through aggressive geometric distortion on every single exposure — for
classes that were *already* hard to tell apart from a close sibling (that's precisely why they
were in list (b) to begin with), the distortion apparently erases enough of the fine
distinguishing detail (cap shape, pore pattern) that the model gives up and defaults to the
sibling's cleaner, undistorted, data-rich decision region instead.


![Mushroom Run 7 (scarcity-aware augmentation) metrics](assets/mushroom_run7_scarcity_aware_metrics.png)


### Where this leaves the mushroom investigation

Five interventions tried against the same ceiling now:

| Intervention | Result |
|---|---|
| Regularization (Run 5: aug+weight_decay+label_smoothing) | Worse |
| Resolution (Run 6: 224px→384px) | **+4.8pt accuracy — the only real win** |
| Architecture (EfficientNet-B3) | Same genus-level mistakes, slightly worse |
| Hierarchical genus+species loss | Marginal gain, confused pairs got worse |
| Scarcity-aware heavy augmentation (Run 7) | Worse — targeted classes collapsed |

**Run 6 (384px, uniform light augmentation, 88.10% val_accuracy / 0.8511 macro-F1) remains the
best mushroom result**, and every attempt to improve on it by changing how the model trains —
more regularization, less regularization but targeted, a different architecture, an auxiliary
loss — has made things worse or produced only a small, partial gain (resolution). Combined with
the Veri Kalitesi Bulguları findings, the pattern is consistent: **the ceiling here is the real
data itself** (110/169 species under 300 real unique photos, 7 species-pairs that are genuinely
hard to separate even with abundant data), not something a training-side intervention can fix.
Further gains would need actual new photographs for the scarce species, not more clever training
— out of scope for this pipeline. Recommend treating Run 6 as the final mushroom result and
moving on (flower is already strong at macro-F1 0.9882; PlantVillage is next per the project
roadmap).


## Run 8/9 — medium-severity calibration and a clean Stage 1 baseline

Two more runs completed, both 384px:

### Calibration (medium augmentation on the same 26 scarce classes)

Follow-up to Run 7 (`heavy`, which collapsed the targeted classes). Testing whether the
*severity* was the problem, not the hypothesis itself, by trying `medium` (±15° rotation,
moderate color jitter, no perspective warp) on the same 26 species.

- val_accuracy: 87.29%, val_macro_f1: 0.8251 (best epoch 29) — **still worse than the uniform-light
  baseline** (Run 6: 88.10%/0.8511), though less damaging than `heavy` (Run 7: 86.59%/0.8029)

| Class | Baseline (light, all classes) | Medium (this run) | Heavy (Run 7) |
|---|---|---|---|
| Boletus reticulatus | 0.552-0.562 | 0.385 | 0.095 |
| Trametes gibbosa | 0.627-0.649 | 0.475 | 0.519 |
| Lycoperdon pyriforme | 0.634-0.659 | 0.492 | 0.200 |
| Daedaleopsis tricolor | 0.684 | 0.500 | 0.250 |
| Amanita persicina | 0.578-0.651 | 0.548 | 0.320 |
| Fomitopsis mounceae | 0.449-0.504 | **0.553 (improved)** | 0.308 |

**Severity scales monotonically with damage** — `light < medium < heavy` in how much each
scarce class's F1 drops, with one exception (Fomitopsis mounceae actually improved slightly
under `medium`). This is a cleaner result than Run 7 alone: it's not that `heavy` specifically
was miscalibrated, it's that *any* amount of extra distortion beyond `light` tends to hurt
already data-thin classes — consistent with the mechanism proposed after Run 7 (distortion
erases fine distinguishing detail that's already scarce). The scarcity-aware-augmentation
hypothesis is now considered tested and not supported, at any severity tried.

### Two-stage fine-tuning — Stage 1 (clean baseline)

A fresh run of Run 6's exact recipe (384px, uniform `light` augmentation, all 169 classes),
under its own `checkpoint_name` so it doesn't collide with other concurrently-running
experiments (see `checkpoint_name` override added to `scripts/train_baseline.py`).

- val_accuracy: 87.96%, val_macro_f1: 0.8475 (best epoch 27)
- Within ~0.1-0.2pt of Run 6 (88.10%/0.8511) — confirms Run 6's result is reproducible within
  normal run-to-run variance (different weight init / data shuffling), not a lucky outlier
- This checkpoint (`mushroom_resnet50_twostage_stage1_best.pt`) is the starting point for
  Stage 2 fine-tuning (below)

## Stage 2 — fine-tuning on scarce classes only (in progress)

Unlike the augmentation experiments above, Stage 2 doesn't add synthetic distortion — it gives
the 26 scarce classes extra, focused gradient updates on their existing (undistorted, `light`-
augmented) images, with most of the backbone frozen (`ResNet50.freeze_except_last_block`: only
`layer4` + `fc` train, 64.2% of parameters) and a low learning rate (1e-5, vs Stage 1's 1e-4) to
limit drift. Evaluated on the *full* validation set every epoch specifically to catch
catastrophic forgetting of the other 143 classes, not just whether the scarce ones improve —
see `scripts/train_stage2_finetune.py`. Launched as job 6114751; result pending.


## Üç katmanlı özet rapor — "%88'in nereden geldiği"

169 sınıfı, genel `%88.10` (Run 6) metriğinin arkasında ne olduğunu şeffaf gösterecek şekilde
üç katmana ayırıyoruz. Veri setine dokunmuyoruz — bu sadece raporlamayı segmentli hale getiriyor.

| Katman | Sınıf sayısı | Anlatı |
|---|---|---|
| **Sağlıklı** | 136 | Beklenen/iyi performans — medyan F1 **0.876**, ortalama F1 0.874. Ana model kalitesinin göstergesi. |
| **Veri kıtlığı (b)** | 26 | Tamamı gerçek mantar türü. "Daha fazla kaynak/veri bulunursa iyileşme potansiyeli var" — açık, model-dışı bir problem. |
| **Görsel ayırt edilemezlik (c)** | 7 tür / **5 essiz çift** | Model tarafında çözülemez — ama nedenler farklı (aşağıda). |

**Not/sınırlama**: Bu ayrım kesin sınırlarla (unique<300 / unique≥1000 eşikleri) çizildiği için
mükemmel temiz değil — "sağlıklı" 136'lık grubun içinde bile **14 sınıf F1<0.80** (en düşüğü
0.5246). Bunlar b/c eşiklerini tam karşılamıyor ama yine de iyileştirilebilir sınırda —
tabloyu okurken "136'nın hepsi mükemmel" denilemeyeceği açık olsun.

### (c) grubunun iki alt-anlatısı

5 essiz çiftin **4'ü mantar-mantar, 1'i liken-liken**:

**Mantar-mantar çiftleri (4)** — Fomes fomentarius↔Fomitopsis pinicola, Fomes fomentarius↔
Fomitopsis betulina, Fomitopsis betulina↔Fomitopsis pinicola, Boletus edulis↔Leccinum scabrum:
Grad-CAM analizimiz (yukarıda) zaten doğrulamıştı — **model doğru yere bakıyor, yine de ayırt
edemiyor**. Bunu "model limitasyonu" değil, "görevin doğal limiti" olarak belgeliyoruz: gerçek
mikolojik ayrım (spor izi rengi, koku, kesit reaksiyonu gibi) muhtemelen fotoğrafta hiç
yakalanmıyor.

**Liken-liken çifti (1)** — Hypogymnia physodes↔Parmelia sulcata: Bu ayrı bir kategori. Mantar
sınıflandırma hedefinin doğrudan bir başarısızlığı değil — iki liken türü zaten taksonomik
olarak birbirine çok yakın, dataset'in "mantar" kapsamına liken dahil edilmiş olmasının
(daha önceki "d) liken/fungi-olmayan" bulgusuyla aynı kök) bir yan etkisi.

### (b) grubu için araştırma yönü (doğrulanmadı, öneri)

26 türün DF20 (Danish Fungi 2020) / FungiTastic gibi harici, açık mantar veri setlerinde kaç
görseli olduğu kontrol edilebilir — bu potansiyel bir iyileştirme yolu olarak not ediliyor,
ama bu oturumda **doğrulanmadı** (gerçek sayıları uydurmamak için bilerek boş bırakıyoruz).
Dataset'e eklemek zorunlu değil, sadece "buradan devam edilebilir" notu bilimsel kapanış
sağlıyor.

### Stage 2 ile bağlantı

Stage 2 (hâlâ çalışıyor, job 6114751) tam olarak bu 26 kişilik (b) grubunu hedefliyor —
sonucu geldiğinde en çok bu grubun F1 değişimine bakılacak, çünkü müdahale doğrudan onlara
yönelik.
